# Ensemble + Reranker v2 — 64GB-safe for the Recommender Systems Challenge

This notebook implements the advanced setup discussed earlier:

1. Chronological validation using `test.csv` as a local public slice by removing its exact rows from `train.csv`.
2. Candidate generators:
   - Global/recent/category popularity
   - EASE<sup>R</sup>
   - RP3β
   - Sparse item-item KNN
   - TF-IDF metadata/content KNN
   - LightGCN ensemble
   - SASRec sequential model
3. Candidate union, with hundreds or thousands of candidates per user.
4. LightGBM LambdaMART reranker with model-score, item, user, and user-item features.
5. Final outputs saved under `outputs/ensemble_reranker_v2_{RUN_ID}/`.

The CSV files are expected in `data/`:
`train.csv`, `test.csv`, `item_meta.csv`, `sample_submission.csv`.

In [1]:
# Optional installs if the environment is missing packages.
# Keep these commented in submitted code unless your README documents them.
# !pip install -q lightgbm torch scikit-learn scipy pandas numpy pyarrow tqdm nbformat

import os
import gc
import json
import math
import time
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception as e:
    HAS_LGB = False
    print("LightGBM import failed. Install with: pip install lightgbm")
    print(repr(e))

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print("PyTorch import failed. Neural generators will be skipped unless installed.")
    print(repr(e))

/home/joris/miniconda3/envs/RS/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------
# Configuration
# -----------------------

SEED = 42
RUN_ID = os.environ.get("RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S"))

DATA_DIR = Path("data")
OUT_DIR = Path(f"outputs/ensemble_reranker_v2_{RUN_ID}")
MODEL_DIR = OUT_DIR / "models"
CACHE_DIR = OUT_DIR / "cache"
for p in [OUT_DIR, MODEL_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
ITEM_META_PATH = DATA_DIR / "item_meta.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

MS_PER_DAY = 1000 * 60 * 60 * 24

# 64GB-safe candidate sizes.
# The previous notebook used 500-900 per generator, which can create 10M-20M+
# candidate rows per fold. These values preserve candidate diversity while keeping
# RAM bounded.
FOLD_MODEL_TOPN = 180
FINAL_MODEL_TOPN = 350

DEFAULT_TOPN_PER_GENERATOR = FOLD_MODEL_TOPN
FINAL_TOPN_PER_GENERATOR = FINAL_MODEL_TOPN

# Before feature engineering for ranker training, keep every retrieved positive
# and only the hardest negative candidates per user.
MAX_NEGATIVES_PER_USER_FOR_RANKER = 250

# Final inference has no labels; keep this many best pre-ranked candidates per user
# before expensive feature engineering.
MAX_FINAL_CANDIDATES_PER_USER = 1200

# Heavy but stronger. Set False for quick debugging.
USE_ROLLING_FOLDS = True

USE_EASE = True
USE_RP3BETA = True
USE_ITEMKNN = True
USE_CONTENTKNN = True
USE_LIGHTGCN = True and HAS_TORCH
USE_SASREC = True and HAS_TORCH

# Strong-PC defaults. Increase epochs after a first successful end-to-end run.
LIGHTGCN_CONFIGS = [
    dict(name="lgcn_d128_l3_s42", dim=128, layers=3, epochs=120, batch_size=4096, lr=2e-3, reg=1e-4, seed=42, topn=700),
    dict(name="lgcn_d256_l3_s43", dim=256, layers=3, epochs=100, batch_size=4096, lr=1.5e-3, reg=1e-4, seed=43, topn=700),
    dict(name="lgcn_d128_l4_s44", dim=128, layers=4, epochs=100, batch_size=4096, lr=2e-3, reg=1e-4, seed=44, topn=700),
]

SASREC_CONFIGS = [
    dict(name="sasrec_d128_l2_s45", hidden_dim=128, n_layers=2, n_heads=2, dropout=0.25,
         max_len=30, epochs=70, batch_size=512, lr=1e-3, seed=45, topn=700),
]

# Two EASE models are usually enough and save ~1.4GB+ RAM versus four dense EASE matrices.
EASE_LAMBDAS = [300.0, 1000.0]

RP3_CONFIGS = [
    dict(name="rp3_a07_b03_k500", alpha=0.7, beta=0.3, topk=500, topn=700),
    dict(name="rp3_a10_b05_k500", alpha=1.0, beta=0.5, topk=500, topn=700),
]

ITEMKNN_CONFIGS = [
    dict(name="itemknn_cos_k500", weighting="cosine", topk=500, topn=700),
    dict(name="itemknn_tfidf_k500", weighting="tfidf", topk=500, topn=700),
]

CONTENT_CONFIGS = [
    dict(name="content_tfidf_k300", topk=250, topn=FOLD_MODEL_TOPN),
]

# Enforce fold-time top-N everywhere. Some configs originally hard-coded 700.
for _cfg in LIGHTGCN_CONFIGS:
    _cfg["topn"] = FOLD_MODEL_TOPN
for _cfg in SASREC_CONFIGS:
    _cfg["topn"] = FOLD_MODEL_TOPN
for _cfg in RP3_CONFIGS:
    _cfg["topn"] = FOLD_MODEL_TOPN
    _cfg["topk"] = min(_cfg.get("topk", 500), 300)
for _cfg in ITEMKNN_CONFIGS:
    _cfg["topn"] = FOLD_MODEL_TOPN
    _cfg["topk"] = min(_cfg.get("topk", 500), 300)

# None means all users in each fold. Set e.g. 2000 for quick debugging.
MAX_VALID_USERS_PER_FOLD = None

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

config_for_log = {
    "run_id": RUN_ID,
    "data_dir": str(DATA_DIR),
    "out_dir": str(OUT_DIR),
    "use_rolling_folds": USE_ROLLING_FOLDS,
    "generators": {
        "ease": USE_EASE,
        "rp3beta": USE_RP3BETA,
        "itemknn": USE_ITEMKNN,
        "contentknn": USE_CONTENTKNN,
        "lightgcn": USE_LIGHTGCN,
        "sasrec": USE_SASREC,
    },
    "lightgcn_configs": LIGHTGCN_CONFIGS,
    "sasrec_configs": SASREC_CONFIGS,
    "ease_lambdas": EASE_LAMBDAS,
    "rp3_configs": RP3_CONFIGS,
    "itemknn_configs": ITEMKNN_CONFIGS,
    "content_configs": CONTENT_CONFIGS,
}
with open(OUT_DIR / "run_config.json", "w") as f:
    json.dump(config_for_log, f, indent=2)

print("RUN_ID:", RUN_ID)
print("OUT_DIR:", OUT_DIR.resolve())

RUN_ID: 20260610_220236
OUT_DIR: /home/joris/Master/Semester 2/Recommender Systems/Final_Assignment_RS/outputs/ensemble_reranker_v2_20260610_220236


In [3]:
# -----------------------
# Load and inspect data
# -----------------------

for p in [TRAIN_PATH, TEST_PATH, ITEM_META_PATH, SAMPLE_SUB_PATH]:
    assert p.exists(), f"Missing {p}. Put the CSV files under {DATA_DIR}/"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
item_meta_raw = pd.read_csv(ITEM_META_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print("Raw shapes")
print("train:", train_raw.shape)
print("test:", test_raw.shape)
print("item_meta:", item_meta_raw.shape)
print("sample_submission:", sample_submission.shape)

display(train_raw.head())
display(test_raw.head())
display(item_meta_raw.head())
display(sample_submission.head())

Raw shapes
train: (162727, 3)
test: (15460, 3)
item_meta: (7603, 15)
sample_submission: (2255, 3)


,item_id,user_id,timestamp
0,7926,22119,1020909887000
1,6719,3664,1072719074000
2,6719,5466,1090094915000
3,6719,16280,1152498722000
4,9677,6383,1161911421000


,item_id,user_id,timestamp
0,4942,3927,1618965129014
1,959,8359,1618965468770
2,9704,1778,1618966728616
3,8483,20375,1618967691412
4,8750,17076,1618969471839


,item_id,main_category,title,average_rating,rating_number,features,description,price,videos,store,categories,details,bought_together,subtitle,author
0,4110,Tools & Home Improvement,"Scotch 6132-BA-10, 75-Inch x 66-Foot x 0.007-I...",4.8,514,['INSULATES AND PROTECTS against abrasion and ...,"[""Scotch Delicate Surface Painter’s Tape is gr...",63.49,[{'title': 'Scotch Vinyl Electrical Tape (2086...,Scotch,"['Industrial & Scientific', 'Adhesives, Sealan...","{'Brand': 'Scotch', 'Color': 'Black', 'Materia...",NaN,NaN,NaN
1,3882,Amazon Home,ChefLand Empty Professional Spray Bottles - Ne...,3.7,327,['DEVELOPED FOR HIGH-LEVEL FUNCTION - our 16-o...,['Get your hands on favorite trigger sprayer b...,8.99,"[{'title': ""You'd be better off putting the sp...",ChefLand,"['Industrial & Scientific', 'Lab & Scientific ...","{'Brand': 'ChefLand', 'Material': 'Plastic', '...",NaN,NaN,NaN
2,10410,Industrial & Scientific,Robert Manufacturing R209 Series Bob Brass Ada...,4.4,429,"['Brass adaptor', 'Adapt male pipe thread stem...",['Brass adaptor. Adapt male pipe thread stems ...,8.00,[{'title': 'DERNORD Brass mini ball valve 1/4 ...,Robert Manufacturing,"['Industrial & Scientific', 'Hydraulics, Pneum...","{'Size': '3/8"" x 1/4""', 'Material': 'Brass', '...",NaN,NaN,NaN
3,6448,Industrial & Scientific,Velleman VTBEND1 Resistor and Axial Component ...,3.6,54,"['Four piece set', 'Make consistent accurate b...",['Resistor and axial component lead bending to...,NaN,[],Velleman,"['Industrial & Scientific', 'Industrial Electr...","{'Is Discontinued By Manufacturer': 'No', 'Pac...",NaN,NaN,NaN
4,11557,Tools & Home Improvement,"Klein Tools VDV526-052 Cable Tester, LAN Scout...",4.6,935,"['Cable tester for twisted pair, data cables t...","[""This compact cable tester is an easy-to-use ...",NaN,"[{'title': 'Klein TraceAll Tone & Probe', 'url...",Klein Tools,"['Industrial & Scientific', 'Test, Measure & I...","{'Brand': 'Klein Tools', 'Power Source': 'Batt...",NaN,NaN,NaN


,ID,user_id,item_id
0,12,12,"6312,12419,6891,664,4243,8377,7962,6635,12842,..."
1,14,14,"7809,5867,9559,3579,8269,2282,4618,2290,12384,..."
2,17,17,"10132,13098,4105,8726,11554,13275,9862,2408,50..."
3,21,21,"11958,1209,11207,5410,7736,9172,1650,5797,7114..."
4,44,44,"10009,10493,3351,9053,7816,7254,8542,4268,1021..."


In [4]:
def basic_interaction_eda(train_df, test_df, sample_sub, item_meta):
    tr = train_df.drop_duplicates(["user_id", "item_id", "timestamp"]).copy()
    te = test_df.drop_duplicates(["user_id", "item_id", "timestamp"]).copy()

    out = {}
    out["train_rows_raw"] = len(train_df)
    out["train_rows_dedup"] = len(tr)
    out["test_rows_raw"] = len(test_df)
    out["test_rows_dedup"] = len(te)
    out["n_train_users"] = tr.user_id.nunique()
    out["n_train_items"] = tr.item_id.nunique()
    out["n_test_users"] = te.user_id.nunique()
    out["n_test_items"] = te.item_id.nunique()
    out["n_submission_users"] = sample_sub.user_id.nunique()
    out["n_meta_items"] = item_meta.item_id.nunique()

    target_users = set(sample_sub.user_id)
    target_hist = tr[tr.user_id.isin(target_users)].groupby("user_id").size()
    out["target_users_with_history"] = int(target_hist.size)
    out["target_hist_mean"] = float(target_hist.mean())
    out["target_hist_median"] = float(target_hist.median())
    out["target_hist_p90"] = float(target_hist.quantile(0.9))
    out["target_hist_max"] = int(target_hist.max())

    meta_items = set(item_meta.item_id)
    out["train_interaction_meta_coverage"] = float(tr.item_id.isin(meta_items).mean())
    out["train_unique_item_meta_coverage"] = float(pd.Series(tr.item_id.unique()).isin(meta_items).mean())
    out["target_interaction_meta_coverage"] = float(tr[tr.user_id.isin(target_users)].item_id.isin(meta_items).mean())

    tr_dt = pd.to_datetime(tr.timestamp, unit="ms", errors="coerce")
    te_dt = pd.to_datetime(te.timestamp, unit="ms", errors="coerce")
    out["train_min_date"] = str(tr_dt.min())
    out["train_max_date"] = str(tr_dt.max())
    out["test_min_date"] = str(te_dt.min())
    out["test_max_date"] = str(te_dt.max())

    exact_overlap = (
        te.merge(
            tr.assign(_in_train=1),
            on=["user_id", "item_id", "timestamp"],
            how="left"
        )["_in_train"]
        .fillna(0)
        .mean()
    )
    out["fraction_test_exactly_in_train"] = float(exact_overlap)

    item_counts = tr.groupby("item_id").size().sort_values(ascending=False)
    for n in [10, 100, 1000, 5000]:
        out[f"top_{n}_items_interaction_share"] = float(item_counts.head(n).sum() / len(tr))

    return pd.Series(out)

eda = basic_interaction_eda(train_raw, test_raw, sample_submission, item_meta_raw)
display(eda.to_frame("value"))
eda.to_json(OUT_DIR / "eda_summary.json", indent=2)

,value
train_rows_raw,162727
train_rows_dedup,158471
test_rows_raw,15460
test_rows_dedup,15158
n_train_users,23284
n_train_items,13441
n_test_users,7437
n_test_items,5984
n_submission_users,2255
n_meta_items,7603


In [5]:
# -----------------------
# Preprocessing and ID encoding
# -----------------------

KEY_COLS = ["user_id", "item_id", "timestamp"]

train = train_raw.drop_duplicates(KEY_COLS).copy()
test = test_raw.drop_duplicates(KEY_COLS).copy()

train["timestamp"] = pd.to_numeric(train["timestamp"], errors="coerce").astype("int64")
test["timestamp"] = pd.to_numeric(test["timestamp"], errors="coerce").astype("int64")

all_user_ids = np.array(sorted(set(train.user_id.unique()) | set(test.user_id.unique()) | set(sample_submission.user_id.unique())))
all_item_ids = np.array(sorted(set(train.item_id.unique()) | set(test.item_id.unique()) | set(item_meta_raw.item_id.unique())))

user2idx = {u: i for i, u in enumerate(all_user_ids)}
idx2user = {i: u for u, i in user2idx.items()}
item2idx = {it: i for i, it in enumerate(all_item_ids)}
idx2item = {i: it for it, i in item2idx.items()}

n_users = len(all_user_ids)
n_items = len(all_item_ids)

def add_indices(df):
    out = df.copy()
    out["user_idx"] = out["user_id"].map(user2idx).astype("int32")
    out["item_idx"] = out["item_id"].map(item2idx).astype("int32")
    return out

train = add_indices(train)
test = add_indices(test)
sample_submission["user_idx"] = sample_submission["user_id"].map(user2idx).astype("int32")

train_item_indices = np.array(sorted(train.item_idx.unique()), dtype=np.int32)
train_item_mask = np.zeros(n_items, dtype=bool)
train_item_mask[train_item_indices] = True

print(f"Encoded users={n_users:,}, items={n_items:,}, train-interaction items={len(train_item_indices):,}")
print("All sample users have mapping:", sample_submission.user_idx.notna().all())

Encoded users=23,284, items=13,441, train-interaction items=13,441
All sample users have mapping: True


In [6]:
def make_interaction_matrix(df, n_users=n_users, n_items=n_items, binary=True):
    if len(df) == 0:
        return sparse.csr_matrix((n_users, n_items), dtype=np.float32)
    rows = df["user_idx"].to_numpy(np.int32)
    cols = df["item_idx"].to_numpy(np.int32)
    vals = np.ones(len(df), dtype=np.float32)
    X = sparse.csr_matrix((vals, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)
    X.sum_duplicates()
    if binary:
        X.data[:] = 1.0
    return X

def split_train_valid_by_exact_test(train_df, test_df):
    valid_keys = test_df[KEY_COLS].drop_duplicates().assign(_valid_row=1)
    merged = train_df.merge(valid_keys, on=KEY_COLS, how="left")
    local_train = merged[merged["_valid_row"].isna()].drop(columns=["_valid_row"]).copy()
    local_valid = test_df.copy()
    warm_users = set(local_train.user_idx.unique())
    local_valid = local_valid[local_valid.user_idx.isin(warm_users)].copy()
    return local_train, local_valid

local_train, local_valid = split_train_valid_by_exact_test(train, test)
print("local_train:", local_train.shape, "local_valid:", local_valid.shape)
print("Validation users:", local_valid.user_idx.nunique())
print("Target-overlap validation users:", local_valid[local_valid.user_idx.isin(sample_submission.user_idx)].user_idx.nunique())

local_train: (143313, 5) local_valid: (15158, 5)
Validation users: 7437
Target-overlap validation users: 1285


In [7]:
# -----------------------
# Static metadata features
# -----------------------

def coerce_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def build_item_static(item_meta, all_item_ids, item2idx):
    meta = item_meta.drop_duplicates("item_id").copy()
    base = pd.DataFrame({"item_id": all_item_ids})
    base["item_idx"] = base["item_id"].map(item2idx).astype("int32")
    base = base.merge(meta, on="item_id", how="left")

    text_cols = ["main_category", "title", "features", "description", "store", "categories", "details", "subtitle"]
    for c in text_cols:
        if c not in base:
            base[c] = ""
        base[c] = base[c].map(coerce_text)

    for c in ["main_category", "store"]:
        values = base[c].replace("", np.nan)
        codes, uniques = pd.factorize(values, sort=True)
        base[f"{c}_code"] = codes.astype("int32")

    for c in ["average_rating", "rating_number", "price"]:
        if c not in base:
            base[c] = np.nan
        base[c] = pd.to_numeric(base[c], errors="coerce")

    base["has_metadata"] = base["title"].ne("").astype("int8")
    base["log_price"] = np.log1p(base["price"].clip(lower=0))
    base["log_rating_number"] = np.log1p(base["rating_number"].fillna(0).clip(lower=0))
    base["title_len"] = base["title"].str.len().fillna(0).astype("float32")
    base["description_len"] = base["description"].str.len().fillna(0).astype("float32")
    base["features_len"] = base["features"].str.len().fillna(0).astype("float32")
    base["categories_len"] = base["categories"].str.len().fillna(0).astype("float32")
    base["text_for_content"] = (
        base["title"] + " " +
        base["main_category"] + " " +
        base["categories"] + " " +
        base["features"] + " " +
        base["description"] + " " +
        base["store"] + " " +
        base["details"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    keep = [
        "item_idx", "item_id", "main_category_code", "store_code", "has_metadata",
        "average_rating", "rating_number", "price", "log_price", "log_rating_number",
        "title_len", "description_len", "features_len", "categories_len",
        "text_for_content"
    ]
    return base[keep].sort_values("item_idx").reset_index(drop=True)

item_static = build_item_static(item_meta_raw, all_item_ids, item2idx)
display(item_static.head())
print("Metadata coverage over encoded catalog:", item_static.has_metadata.mean())

,item_idx,item_id,main_category_code,store_code,has_metadata,average_rating,rating_number,price,log_price,log_rating_number,title_len,description_len,features_len,categories_len,text_for_content
0,0,1,-1,-1,0,NaN,NaN,NaN,NaN,0.00000,0.0,0.0,0.0,0.0,
1,1,2,-1,-1,0,NaN,NaN,NaN,NaN,0.00000,0.0,0.0,0.0,0.0,
2,2,3,-1,-1,0,NaN,NaN,NaN,NaN,0.00000,0.0,0.0,0.0,0.0,
3,3,4,-1,-1,0,NaN,NaN,NaN,NaN,0.00000,0.0,0.0,0.0,0.0,
4,4,5,16,1569,1,4.4,162.0,19.99,3.044046,5.09375,119.0,683.0,562.0,128.0,MCIGICM 10pcs Breadboard 830 Point Solderless ...


Metadata coverage over encoded catalog: 0.5656573171639014


In [8]:
# -----------------------
# Evaluation helpers
# -----------------------

def recall_at_k_from_ranked(ranked_df, truth_df, k=10, users=None, score_col="score"):
    if users is None:
        users = sorted(truth_df.user_idx.unique())
    else:
        users = list(users)

    truth = truth_df.groupby("user_idx")["item_idx"].apply(lambda x: set(x.values)).to_dict()
    pred_top = (
        ranked_df.sort_values(["user_idx", score_col], ascending=[True, False])
        .groupby("user_idx")
        .head(k)
        .groupby("user_idx")["item_idx"]
        .apply(list)
        .to_dict()
    )
    recalls = []
    for u in users:
        t = truth.get(u, set())
        if not t:
            continue
        p = pred_top.get(u, [])
        recalls.append(len(set(p[:k]) & t) / min(k, len(t)))
    return float(np.mean(recalls)) if recalls else np.nan

def save_df(df, path):
    path = Path(path)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception:
        csv_path = path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        return csv_path

def timestamp_ms(date_str):
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

In [9]:
# -----------------------
# Sparse top-K and candidate utilities
# -----------------------

@dataclass
class CandidateModel:
    name: str
    topn: int
    score_batch_fn: object
    similarity_matrix: object = None
    use_similarity_features: bool = False

    def score_batch(self, user_indices):
        return self.score_batch_fn(user_indices)

def topk_candidates_from_score_fn(model, user_indices, X_seen, allowed_items, chunk_size=256):
    rows = []
    allowed_items = np.asarray(allowed_items, dtype=np.int32)
    n_allowed = len(allowed_items)
    if n_allowed == 0:
        return pd.DataFrame(columns=["user_idx", "item_idx", "score", "rank", "model"])

    k = min(model.topn, n_allowed)
    user_indices = np.asarray(user_indices, dtype=np.int32)

    for start in tqdm(range(0, len(user_indices), chunk_size), desc=f"candidates:{model.name}", leave=False):
        batch_users = user_indices[start:start + chunk_size]
        scores = model.score_batch(batch_users)
        if sparse.issparse(scores):
            scores = scores.toarray()
        scores = np.asarray(scores, dtype=np.float32)

        for r, u in enumerate(batch_users):
            s_allowed = scores[r, allowed_items].copy()
            seen = X_seen[u].indices
            if len(seen):
                seen_mask = np.isin(allowed_items, seen, assume_unique=False)
                s_allowed[seen_mask] = -np.inf

            finite = np.isfinite(s_allowed)
            if not np.any(finite):
                continue
            valid_pos = np.where(finite)[0]
            valid_scores = s_allowed[valid_pos]

            if len(valid_pos) > k:
                part = np.argpartition(-valid_scores, k - 1)[:k]
                top_pos = valid_pos[part]
                top_scores = valid_scores[part]
            else:
                top_pos = valid_pos
                top_scores = valid_scores

            order = np.argsort(-top_scores)
            top_items = allowed_items[top_pos[order]]
            top_scores = top_scores[order]

            for rank, (it, sc) in enumerate(zip(top_items, top_scores), start=1):
                rows.append((int(u), int(it), float(sc), int(rank), model.name))

    return pd.DataFrame(rows, columns=["user_idx", "item_idx", "score", "rank", "model"])

def combine_candidate_frames(frames):
    frames = [f for f in frames if f is not None and len(f) > 0]
    if not frames:
        return pd.DataFrame(columns=["user_idx", "item_idx"])

    # With 64GB RAM, the long concatenated table is the most dangerous object.
    # Keep top-N modest in config, and explicitly delete temporaries below.
    long = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()

    long = long.drop_duplicates(["user_idx", "item_idx", "model"], keep="first")

    base = (
        long.groupby(["user_idx", "item_idx"])
        .agg(
            n_retrievers=("model", "nunique"),
            best_rank=("rank", "min"),
            mean_rank=("rank", "mean"),
        )
        .reset_index()
    )
    rr = (
        long.assign(rr=1.0 / long["rank"].clip(lower=1))
        .groupby(["user_idx", "item_idx"])["rr"]
        .agg(["sum", "mean", "max"])
        .rename(columns={"sum": "sum_rr", "mean": "mean_rr", "max": "max_rr"})
        .reset_index()
    )
    base = base.merge(rr, on=["user_idx", "item_idx"], how="left")

    score_wide = long.pivot_table(
        index=["user_idx", "item_idx"], columns="model", values="score", aggfunc="max"
    )
    score_wide.columns = [f"{c}__score" for c in score_wide.columns]
    score_wide = score_wide.reset_index()

    rank_wide = long.pivot_table(
        index=["user_idx", "item_idx"], columns="model", values="rank", aggfunc="min"
    )
    rank_wide.columns = [f"{c}__rank" for c in rank_wide.columns]
    rank_wide = rank_wide.reset_index()

    del long
    gc.collect()

    out = base.merge(score_wide, on=["user_idx", "item_idx"], how="left")
    out = out.merge(rank_wide, on=["user_idx", "item_idx"], how="left")
    del base, score_wide, rank_wide
    gc.collect()
    return out

def add_candidate_prescore(df):
    return (
        df["sum_rr"].fillna(0).astype("float32")
        + 0.05 * df["n_retrievers"].fillna(0).astype("float32")
        - 1e-6 * df["best_rank"].fillna(1_000_000).astype("float32")
    )

def prefilter_candidates_for_fold(candidates, positives, max_neg_per_user=250):
    """
    Memory-safe training fold pruning.
    Preserve every retrieved positive, then keep only the hardest negatives per user.
    Do this before expensive feature engineering.
    """
    before = len(candidates)
    candidates = candidates.copy()
    candidates["_label_tmp"] = [
        1 if (int(u), int(i)) in positives else 0
        for u, i in zip(candidates["user_idx"], candidates["item_idx"])
    ]
    candidates["_pre_score"] = add_candidate_prescore(candidates)

    pos = candidates[candidates["_label_tmp"] == 1]
    neg = candidates[candidates["_label_tmp"] == 0]

    neg = (
        neg.sort_values(["user_idx", "_pre_score"], ascending=[True, False])
        .groupby("user_idx", sort=False)
        .head(max_neg_per_user)
    )

    out = (
        pd.concat([pos, neg], ignore_index=True)
        .drop(columns=["_label_tmp", "_pre_score"])
        .reset_index(drop=True)
    )
    print(
        f"Candidate prefilter for ranker: {before:,} -> {len(out):,} rows "
        f"(kept positives={len(pos):,}, max_neg/user={max_neg_per_user})"
    )
    del candidates, pos, neg
    gc.collect()
    return out

def prefilter_candidates_for_final(candidates, max_per_user=1200):
    """
    Memory-safe final inference pruning.
    No labels are available, so keep candidates with the best retrieval consensus.
    """
    before = len(candidates)
    candidates = candidates.copy()
    candidates["_pre_score"] = add_candidate_prescore(candidates)
    out = (
        candidates.sort_values(["user_idx", "_pre_score"], ascending=[True, False])
        .groupby("user_idx", sort=False)
        .head(max_per_user)
        .drop(columns=["_pre_score"])
        .reset_index(drop=True)
    )
    print(f"Final candidate prefilter: {before:,} -> {len(out):,} rows (max/user={max_per_user})")
    del candidates
    gc.collect()
    return out

In [10]:
# -----------------------
# Popularity candidate models
# -----------------------

def time_weighted_item_scores(df, half_life_days=None, since_ts=None, n_items=n_items):
    if since_ts is not None:
        df = df[df["timestamp"] >= since_ts]
    if len(df) == 0:
        return np.zeros(n_items, dtype=np.float32)

    if half_life_days is None:
        weights = np.ones(len(df), dtype=np.float32)
    else:
        max_ts = df["timestamp"].max()
        age_days = (max_ts - df["timestamp"].to_numpy()) / MS_PER_DAY
        weights = np.exp(-np.log(2) * age_days / half_life_days).astype(np.float32)

    scores = np.bincount(df["item_idx"].to_numpy(np.int32), weights=weights, minlength=n_items).astype(np.float32)
    return np.log1p(scores)

class CategoryPopularityModel:
    def __init__(self, name, train_df, item_static, topn=500, half_life_days=365):
        self.name = name
        self.topn = topn
        self.item_main_cat = item_static.set_index("item_idx")["main_category_code"].reindex(range(n_items)).fillna(-1).astype("int32").to_numpy()
        self.global_scores = time_weighted_item_scores(train_df, half_life_days=half_life_days)
        self.user_dom_cat = self._user_dominant_category(train_df)
        self.cat_scores = {}
        for c in np.unique(self.item_main_cat):
            if c < 0:
                continue
            mask = self.item_main_cat == c
            s = np.zeros(n_items, dtype=np.float32)
            s[mask] = self.global_scores[mask]
            self.cat_scores[int(c)] = s

    def _user_dominant_category(self, train_df):
        tmp = train_df[["user_idx", "item_idx"]].copy()
        tmp["cat"] = self.item_main_cat[tmp["item_idx"].values]
        tmp = tmp[tmp["cat"] >= 0]
        if len(tmp) == 0:
            return {}
        counts = tmp.groupby(["user_idx", "cat"]).size().reset_index(name="cnt")
        counts = counts.sort_values(["user_idx", "cnt"], ascending=[True, False])
        return counts.drop_duplicates("user_idx").set_index("user_idx")["cat"].astype("int32").to_dict()

    def score_batch(self, user_indices):
        out = np.empty((len(user_indices), n_items), dtype=np.float32)
        for r, u in enumerate(user_indices):
            c = self.user_dom_cat.get(int(u), -1)
            if c in self.cat_scores:
                out[r] = 0.25 * self.global_scores + 1.25 * self.cat_scores[c]
            else:
                out[r] = self.global_scores
        return out

def build_popularity_models(train_df, item_static, topn=500):
    models = []
    specs = [
        ("pop_global", None, None),
        ("pop_decay_180d", 180, None),
        ("pop_decay_365d", 365, None),
        ("pop_recent_365d", None, train_df["timestamp"].max() - 365 * MS_PER_DAY),
        ("pop_recent_730d", None, train_df["timestamp"].max() - 730 * MS_PER_DAY),
    ]
    for name, half_life, since_ts in specs:
        scores = time_weighted_item_scores(train_df, half_life_days=half_life, since_ts=since_ts)
        models.append(CandidateModel(
            name=name,
            topn=topn,
            score_batch_fn=lambda users, scores=scores: np.tile(scores, (len(users), 1))
        ))

    cat = CategoryPopularityModel("pop_category_365d", train_df, item_static, topn=topn, half_life_days=365)
    models.append(CandidateModel(cat.name, cat.topn, cat.score_batch))
    return models

In [11]:
# -----------------------
# EASE^R
# -----------------------

class EASEModel:
    def __init__(self, lam=300.0, name=None, topn=700):
        self.lam = float(lam)
        self.name = name or f"ease_lam{int(lam)}"
        self.topn = topn
        self.B = None
        self.X = None

    def fit(self, X):
        print(f"Fitting {self.name}: X={X.shape}, nnz={X.nnz:,}, lambda={self.lam}")
        self.X = X.tocsr().astype(np.float32)
        G = (self.X.T @ self.X).toarray().astype(np.float64)
        diag_idx = np.diag_indices(G.shape[0])
        G[diag_idx] += self.lam
        P = np.linalg.inv(G)
        B = -P / np.diag(P)
        B[diag_idx] = 0.0
        self.B = B.astype(np.float32)
        del G, P
        gc.collect()
        return self

    def score_batch(self, user_indices):
        return self.X[user_indices] @ self.B

def build_ease_models(X, lambdas=EASE_LAMBDAS, topn=700):
    models = []
    for lam in lambdas:
        m = EASEModel(lam=lam, topn=topn).fit(X)
        models.append(CandidateModel(m.name, m.topn, m.score_batch))
    return models

In [12]:
# -----------------------
# RP3β and item-item KNN
# -----------------------

def topk_sparse_rows(M, k=500):
    M = M.tocsr()
    rows, cols, data = [], [], []
    for i in tqdm(range(M.shape[0]), desc=f"topk_sparse_rows(k={k})", leave=False):
        start, end = M.indptr[i], M.indptr[i + 1]
        row_cols = M.indices[start:end]
        row_data = M.data[start:end]
        if len(row_data) == 0:
            continue
        if len(row_data) > k:
            idx = np.argpartition(-row_data, k - 1)[:k]
            row_cols = row_cols[idx]
            row_data = row_data[idx]
        order = np.argsort(-row_data)
        row_cols = row_cols[order]
        row_data = row_data[order]
        rows.extend([i] * len(row_cols))
        cols.extend(row_cols.tolist())
        data.extend(row_data.astype(np.float32).tolist())
    out = sparse.csr_matrix((data, (rows, cols)), shape=M.shape, dtype=np.float32)
    out.eliminate_zeros()
    return out

def fit_rp3beta_similarity(X, alpha=0.7, beta=0.3, topk=500):
    print(f"Fitting RP3beta alpha={alpha}, beta={beta}, topk={topk}")
    X = X.tocsr().astype(np.float32)

    Pui = normalize(X, norm="l1", axis=1)
    Piu = normalize(X.T, norm="l1", axis=1)

    if alpha != 1.0:
        Pui = Pui.copy()
        Piu = Piu.copy()
        Pui.data = np.power(Pui.data, alpha).astype(np.float32)
        Piu.data = np.power(Piu.data, alpha).astype(np.float32)

    S = Piu @ Pui
    S = S.tolil()
    S.setdiag(0)
    S = S.tocsr()
    S.eliminate_zeros()

    item_degree = np.asarray(X.sum(axis=0)).ravel().astype(np.float32)
    degree_penalty = np.power(item_degree + 1e-6, -beta).astype(np.float32)
    S = S.multiply(degree_penalty.reshape(1, -1)).tocsr()
    S = topk_sparse_rows(S, k=topk)
    return S

def fit_itemknn_similarity(X, weighting="cosine", topk=500):
    print(f"Fitting ItemKNN weighting={weighting}, topk={topk}")
    X = X.tocsr().astype(np.float32)

    if weighting == "tfidf":
        Xw = X.copy().astype(np.float32)
        df = np.diff(Xw.tocsc().indptr).astype(np.float32)
        idf = np.log((Xw.shape[0] + 1.0) / (df + 1.0)) + 1.0
        Xw = Xw @ sparse.diags(idf)
    else:
        Xw = X

    Y = normalize(Xw.T, norm="l2", axis=1)
    S = Y @ Y.T
    S = S.tolil()
    S.setdiag(0)
    S = S.tocsr()
    S.eliminate_zeros()
    S = topk_sparse_rows(S, k=topk)
    return S

class SparseLinearScoreModel:
    def __init__(self, name, X, W, topn=700, use_similarity_features=True):
        self.name = name
        self.X = X.tocsr()
        self.W = W.tocsr()
        self.topn = topn
        self.use_similarity_features = use_similarity_features

    def score_batch(self, user_indices):
        return self.X[user_indices] @ self.W

def build_graph_item_models(X):
    models = []

    if USE_RP3BETA:
        for cfg in RP3_CONFIGS:
            W = fit_rp3beta_similarity(X, alpha=cfg["alpha"], beta=cfg["beta"], topk=cfg["topk"])
            sm = SparseLinearScoreModel(cfg["name"], X, W, topn=cfg["topn"], use_similarity_features=True)
            models.append(CandidateModel(sm.name, sm.topn, sm.score_batch, similarity_matrix=sm.W, use_similarity_features=True))

    if USE_ITEMKNN:
        for cfg in ITEMKNN_CONFIGS:
            W = fit_itemknn_similarity(X, weighting=cfg["weighting"], topk=cfg["topk"])
            sm = SparseLinearScoreModel(cfg["name"], X, W, topn=cfg["topn"], use_similarity_features=True)
            models.append(CandidateModel(sm.name, sm.topn, sm.score_batch, similarity_matrix=sm.W, use_similarity_features=True))

    return models

In [13]:
# -----------------------
# Content KNN from item metadata only
# -----------------------

def fit_content_similarity(item_static, allowed_items, topk=300):
    print(f"Fitting content KNN topk={topk}")
    df = item_static.copy()
    df = df[df["item_idx"].isin(allowed_items)].copy()
    df["text_for_content"] = df["text_for_content"].fillna("").astype(str)
    df = df[df["text_for_content"].str.len() > 0].copy()
    if len(df) == 0:
        return sparse.csr_matrix((n_items, n_items), dtype=np.float32)

    vectorizer = FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=250_000,
            sublinear_tf=True,
            dtype=np.float32,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=150_000,
            sublinear_tf=True,
            dtype=np.float32,
        )),
    ])

    Xtxt = vectorizer.fit_transform(df["text_for_content"])
    Xtxt = normalize(Xtxt, norm="l2", axis=1)

    k = min(topk + 1, Xtxt.shape[0])
    nn = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute", n_jobs=-1)
    nn.fit(Xtxt)
    dist, neigh = nn.kneighbors(Xtxt, return_distance=True)

    global_items = df["item_idx"].to_numpy(np.int32)
    rows, cols, data = [], [], []
    for local_i in tqdm(range(len(global_items)), desc="content neighbors", leave=False):
        src = int(global_items[local_i])
        for d, local_j in zip(dist[local_i], neigh[local_i]):
            tgt = int(global_items[local_j])
            if src == tgt:
                continue
            sim = 1.0 - float(d)
            if sim <= 0:
                continue
            rows.append(src)
            cols.append(tgt)
            data.append(sim)

    W = sparse.csr_matrix((np.array(data, dtype=np.float32), (rows, cols)), shape=(n_items, n_items), dtype=np.float32)
    W = topk_sparse_rows(W, k=topk)
    return W

def build_content_models(X, item_static, allowed_items):
    models = []
    if USE_CONTENTKNN:
        for cfg in CONTENT_CONFIGS:
            W = fit_content_similarity(item_static, allowed_items, topk=cfg["topk"])
            sm = SparseLinearScoreModel(cfg["name"], X, W, topn=cfg["topn"], use_similarity_features=True)
            models.append(CandidateModel(sm.name, sm.topn, sm.score_batch, similarity_matrix=sm.W, use_similarity_features=True))
    return models

In [14]:
# -----------------------
# LightGCN
# -----------------------

if HAS_TORCH:
    class TorchLightGCN(nn.Module):
        def __init__(self, n_users, n_items, dim=128, n_layers=3):
            super().__init__()
            self.n_users = n_users
            self.n_items = n_items
            self.dim = dim
            self.n_layers = n_layers
            self.user_emb = nn.Embedding(n_users, dim)
            self.item_emb = nn.Embedding(n_items, dim)
            nn.init.normal_(self.user_emb.weight, std=0.1)
            nn.init.normal_(self.item_emb.weight, std=0.1)

        def propagate(self, norm_adj):
            all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
            embs = [all_emb]
            for _ in range(self.n_layers):
                all_emb = torch.sparse.mm(norm_adj, all_emb)
                embs.append(all_emb)
            final = torch.stack(embs, dim=0).mean(dim=0)
            users, items = torch.split(final, [self.n_users, self.n_items], dim=0)
            return users, items

    def build_lightgcn_adj(X, device):
        Xcoo = X.tocoo()
        u = Xcoo.row.astype(np.int64)
        i = Xcoo.col.astype(np.int64) + n_users
        rows = np.concatenate([u, i])
        cols = np.concatenate([i, u])

        deg = np.bincount(rows, minlength=n_users + n_items).astype(np.float32)
        deg_inv_sqrt = np.power(deg + 1e-8, -0.5)
        vals = deg_inv_sqrt[rows] * deg_inv_sqrt[cols]

        idx = torch.LongTensor(np.vstack([rows, cols]))
        vals = torch.FloatTensor(vals)
        adj = torch.sparse_coo_tensor(idx, vals, size=(n_users + n_items, n_users + n_items))
        return adj.coalesce().to(device)

    def prepare_user_positive_arrays(X):
        X = X.tocsr()
        active_users = np.where(np.diff(X.indptr) > 0)[0].astype(np.int32)
        pos_arrays = {}
        pos_sets = {}
        for u in active_users:
            arr = X[u].indices.astype(np.int32)
            pos_arrays[int(u)] = arr
            pos_sets[int(u)] = set(arr.tolist())
        return active_users, pos_arrays, pos_sets

    def sample_bpr_batch(active_users, pos_arrays, pos_sets, n_items, batch_size, rng):
        users = rng.choice(active_users, size=batch_size, replace=True)
        pos = np.empty(batch_size, dtype=np.int64)
        neg = rng.integers(0, n_items, size=batch_size, dtype=np.int64)
        for r, u in enumerate(users):
            positives = pos_arrays[int(u)]
            pos[r] = positives[rng.integers(0, len(positives))]
        for r, u in enumerate(users):
            seen = pos_sets[int(u)]
            tries = 0
            while int(neg[r]) in seen and tries < 50:
                neg[r] = rng.integers(0, n_items)
                tries += 1
        return users.astype(np.int64), pos, neg

    class LightGCNScoreModel:
        def __init__(self, cfg):
            self.cfg = cfg
            self.name = cfg["name"]
            self.topn = cfg.get("topn", 700)
            self.user_emb = None
            self.item_emb = None
            self.device = None

        def fit(self, X):
            seed_everything(self.cfg.get("seed", SEED))
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.device = device
            print(f"Training {self.name} on {device}")

            active_users, pos_arrays, pos_sets = prepare_user_positive_arrays(X)
            norm_adj = build_lightgcn_adj(X, device)

            model = TorchLightGCN(
                n_users=n_users,
                n_items=n_items,
                dim=self.cfg.get("dim", 128),
                n_layers=self.cfg.get("layers", 3),
            ).to(device)

            opt = torch.optim.Adam(model.parameters(), lr=self.cfg.get("lr", 2e-3))
            batch_size = self.cfg.get("batch_size", 4096)
            epochs = self.cfg.get("epochs", 100)
            reg = self.cfg.get("reg", 1e-4)
            rng = np.random.default_rng(self.cfg.get("seed", SEED))
            steps_per_epoch = max(1, X.nnz // batch_size)

            for epoch in tqdm(range(1, epochs + 1), desc=self.name):
                model.train()
                total_loss = 0.0
                for _ in range(steps_per_epoch):
                    users, pos, neg = sample_bpr_batch(active_users, pos_arrays, pos_sets, n_items, batch_size, rng)
                    users_t = torch.LongTensor(users).to(device)
                    pos_t = torch.LongTensor(pos).to(device)
                    neg_t = torch.LongTensor(neg).to(device)

                    user_e, item_e = model.propagate(norm_adj)
                    u_e = user_e[users_t]
                    p_e = item_e[pos_t]
                    n_e = item_e[neg_t]

                    pos_scores = (u_e * p_e).sum(dim=1)
                    neg_scores = (u_e * n_e).sum(dim=1)
                    mf_loss = -F.logsigmoid(pos_scores - neg_scores).mean()
                    reg_loss = reg * (
                        model.user_emb(users_t).norm(2).pow(2) +
                        model.item_emb(pos_t).norm(2).pow(2) +
                        model.item_emb(neg_t).norm(2).pow(2)
                    ) / batch_size
                    loss = mf_loss + reg_loss

                    opt.zero_grad()
                    loss.backward()
                    opt.step()
                    total_loss += float(loss.detach().cpu())

                if epoch % 20 == 0 or epoch == 1:
                    print(f"{self.name} epoch {epoch:03d}/{epochs}, loss={total_loss / steps_per_epoch:.5f}")

            model.eval()
            with torch.no_grad():
                user_e, item_e = model.propagate(norm_adj)
                self.user_emb = user_e.detach().cpu().numpy().astype(np.float32)
                self.item_emb = item_e.detach().cpu().numpy().astype(np.float32)

            del model, norm_adj
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            return self

        def score_batch(self, user_indices):
            return self.user_emb[user_indices] @ self.item_emb.T

def build_lightgcn_models(X):
    models = []
    if USE_LIGHTGCN:
        for cfg in LIGHTGCN_CONFIGS:
            m = LightGCNScoreModel(cfg).fit(X)
            models.append(CandidateModel(m.name, m.topn, m.score_batch))
    return models

In [15]:
# -----------------------
# SASRec sequential generator
# -----------------------

if HAS_TORCH:
    class SASRecDataset(Dataset):
        def __init__(self, train_df, max_len, n_items, seed=SEED, max_examples_per_user=None):
            self.max_len = max_len
            self.n_items = n_items
            self.rng = np.random.default_rng(seed)

            seqs = (
                train_df.sort_values(["user_idx", "timestamp"])
                .groupby("user_idx")["item_idx"]
                .apply(lambda x: x.to_numpy(np.int32))
                .to_dict()
            )
            self.seqs = seqs
            self.user_seen = {int(u): set(arr.tolist()) for u, arr in seqs.items()}
            self.examples = []
            for u, arr in seqs.items():
                if len(arr) < 2:
                    continue
                positions = list(range(1, len(arr)))
                if max_examples_per_user is not None and len(positions) > max_examples_per_user:
                    positions = positions[-max_examples_per_user:]
                for t in positions:
                    self.examples.append((int(u), int(t)))

        def __len__(self):
            return len(self.examples)

        def __getitem__(self, idx):
            u, t = self.examples[idx]
            arr = self.seqs[u]
            prefix = arr[max(0, t - self.max_len):t]
            target = int(arr[t])
            seq = np.zeros(self.max_len, dtype=np.int64)
            seq[-len(prefix):] = prefix + 1

            neg = self.rng.integers(0, self.n_items)
            seen = self.user_seen[u]
            tries = 0
            while int(neg) in seen and tries < 50:
                neg = self.rng.integers(0, self.n_items)
                tries += 1
            return torch.LongTensor(seq), torch.LongTensor([target + 1]).squeeze(0), torch.LongTensor([int(neg) + 1]).squeeze(0)

    class SASRecNet(nn.Module):
        def __init__(self, n_items, max_len=30, hidden_dim=128, n_layers=2, n_heads=2, dropout=0.25):
            super().__init__()
            self.n_items = n_items
            self.max_len = max_len
            self.item_emb = nn.Embedding(n_items + 1, hidden_dim, padding_idx=0)
            self.pos_emb = nn.Embedding(max_len, hidden_dim)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=n_heads,
                dim_feedforward=hidden_dim * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
            self.dropout = nn.Dropout(dropout)
            self.norm = nn.LayerNorm(hidden_dim)
            nn.init.normal_(self.item_emb.weight, std=0.02)
            nn.init.normal_(self.pos_emb.weight, std=0.02)

        def forward(self, seq):
            B, L = seq.shape
            positions = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
            x = self.item_emb(seq) + self.pos_emb(positions)
            x = self.dropout(x)

            causal_mask = torch.triu(torch.ones(L, L, device=seq.device), diagonal=1).bool()
            padding_mask = seq.eq(0)
            out = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)
            out = self.norm(out)

            lengths = seq.ne(0).sum(dim=1).clamp(min=1) - 1
            h = out[torch.arange(B, device=seq.device), lengths]
            return h

    class SASRecScoreModel:
        def __init__(self, cfg):
            self.cfg = cfg
            self.name = cfg["name"]
            self.topn = cfg.get("topn", 700)
            self.max_len = cfg.get("max_len", 30)
            self.model = None
            self.device = None
            self.user_sequences = None
            self.item_weight = None

        def fit(self, train_df):
            seed_everything(self.cfg.get("seed", SEED))
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.device = device
            print(f"Training {self.name} on {device}")

            dataset = SASRecDataset(train_df, max_len=self.max_len, n_items=n_items, seed=self.cfg.get("seed", SEED))
            loader = DataLoader(dataset, batch_size=self.cfg.get("batch_size", 512), shuffle=True, num_workers=0, drop_last=False)

            model = SASRecNet(
                n_items=n_items,
                max_len=self.max_len,
                hidden_dim=self.cfg.get("hidden_dim", 128),
                n_layers=self.cfg.get("n_layers", 2),
                n_heads=self.cfg.get("n_heads", 2),
                dropout=self.cfg.get("dropout", 0.25),
            ).to(device)

            opt = torch.optim.AdamW(model.parameters(), lr=self.cfg.get("lr", 1e-3), weight_decay=1e-5)
            epochs = self.cfg.get("epochs", 50)

            for epoch in tqdm(range(1, epochs + 1), desc=self.name):
                model.train()
                total = 0.0
                for seq, pos, neg in loader:
                    seq = seq.to(device)
                    pos = pos.to(device)
                    neg = neg.to(device)

                    h = model(seq)
                    pos_e = model.item_emb(pos)
                    neg_e = model.item_emb(neg)
                    pos_scores = (h * pos_e).sum(dim=1)
                    neg_scores = (h * neg_e).sum(dim=1)
                    loss = -F.logsigmoid(pos_scores - neg_scores).mean()

                    opt.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    opt.step()
                    total += float(loss.detach().cpu())

                if epoch % 10 == 0 or epoch == 1:
                    print(f"{self.name} epoch {epoch:03d}/{epochs}, loss={total / max(1, len(loader)):.5f}")

            self.model = model.eval()
            self.item_weight = self.model.item_emb.weight.detach().cpu().numpy()[1:].astype(np.float32)
            self.user_sequences = (
                train_df.sort_values(["user_idx", "timestamp"])
                .groupby("user_idx")["item_idx"]
                .apply(lambda x: x.to_numpy(np.int32))
                .to_dict()
            )
            return self

        def _make_seq_batch(self, user_indices):
            seqs = np.zeros((len(user_indices), self.max_len), dtype=np.int64)
            for r, u in enumerate(user_indices):
                hist = self.user_sequences.get(int(u), np.array([], dtype=np.int32))
                hist = hist[-self.max_len:]
                if len(hist):
                    seqs[r, -len(hist):] = hist + 1
            return seqs

        def score_batch(self, user_indices):
            self.model.eval()
            out_scores = []
            bs = 512
            with torch.no_grad():
                for start in range(0, len(user_indices), bs):
                    batch = user_indices[start:start + bs]
                    seq = torch.LongTensor(self._make_seq_batch(batch)).to(self.device)
                    h = self.model(seq)
                    item_weight = self.model.item_emb.weight[1:]
                    scores = h @ item_weight.T
                    out_scores.append(scores.detach().cpu().numpy().astype(np.float32))
            return np.vstack(out_scores)

def build_sasrec_models(train_df):
    models = []
    if USE_SASREC:
        for cfg in SASREC_CONFIGS:
            m = SASRecScoreModel(cfg).fit(train_df)
            models.append(CandidateModel(m.name, m.topn, m.score_batch))
    return models

In [16]:
# -----------------------
# Feature engineering
# -----------------------

def build_item_context_features(train_df, item_static):
    max_ts = train_df["timestamp"].max()
    item_stats = (
        train_df.groupby("item_idx")
        .agg(
            item_count=("user_idx", "size"),
            item_user_count=("user_idx", "nunique"),
            item_first_ts=("timestamp", "min"),
            item_last_ts=("timestamp", "max"),
            item_mean_ts=("timestamp", "mean"),
        )
        .reset_index()
    )

    def recent_counts(days):
        cutoff = max_ts - days * MS_PER_DAY
        return (
            train_df[train_df["timestamp"] >= cutoff]
            .groupby("item_idx")
            .size()
            .rename(f"item_count_last_{days}d")
            .reset_index()
        )

    item_feat = item_static.drop(columns=["text_for_content"]).copy()
    item_feat = item_feat.merge(item_stats, on="item_idx", how="left")
    for days in [90, 180, 365, 730]:
        item_feat = item_feat.merge(recent_counts(days), on="item_idx", how="left")

    count_cols = [c for c in item_feat.columns if c.startswith("item_count")]
    for c in count_cols + ["item_user_count"]:
        if c in item_feat:
            item_feat[c] = item_feat[c].fillna(0).astype("float32")
            item_feat[f"log1p_{c}"] = np.log1p(item_feat[c]).astype("float32")

    item_feat["item_age_days"] = ((max_ts - item_feat["item_first_ts"]) / MS_PER_DAY).astype("float32")
    item_feat["item_days_since_last"] = ((max_ts - item_feat["item_last_ts"]) / MS_PER_DAY).astype("float32")
    item_feat["item_mean_age_days"] = ((max_ts - item_feat["item_mean_ts"]) / MS_PER_DAY).astype("float32")

    item_feat["item_trend_90_vs_730"] = (
        (item_feat.get("item_count_last_90d", 0).fillna(0) + 1.0) /
        (item_feat.get("item_count_last_730d", 0).fillna(0) + 1.0)
    ).astype("float32")
    item_feat["item_trend_180_vs_total"] = (
        (item_feat.get("item_count_last_180d", 0).fillna(0) + 1.0) /
        (item_feat.get("item_count", 0).fillna(0) + 1.0)
    ).astype("float32")

    return item_feat

def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    if counts.sum() <= 0:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p + 1e-12)).sum())

def build_user_context_features(train_df, item_static):
    max_ts = train_df["timestamp"].max()
    user_stats = (
        train_df.groupby("user_idx")
        .agg(
            user_history_len=("item_idx", "size"),
            user_unique_items=("item_idx", "nunique"),
            user_first_ts=("timestamp", "min"),
            user_last_ts=("timestamp", "max"),
            user_mean_ts=("timestamp", "mean"),
        )
        .reset_index()
    )
    user_stats["user_span_days"] = ((user_stats["user_last_ts"] - user_stats["user_first_ts"]) / MS_PER_DAY).astype("float32")
    user_stats["user_days_since_last"] = ((max_ts - user_stats["user_last_ts"]) / MS_PER_DAY).astype("float32")
    user_stats["user_mean_age_days"] = ((max_ts - user_stats["user_mean_ts"]) / MS_PER_DAY).astype("float32")

    item_small = item_static[["item_idx", "main_category_code", "store_code", "price", "average_rating"]]
    tmp = train_df[["user_idx", "item_idx"]].merge(item_small, on="item_idx", how="left")

    cat_counts = tmp[tmp["main_category_code"] >= 0].groupby(["user_idx", "main_category_code"]).size().reset_index(name="cnt")
    if len(cat_counts):
        cat_counts = cat_counts.sort_values(["user_idx", "cnt"], ascending=[True, False])
        dom_cat = cat_counts.drop_duplicates("user_idx").rename(columns={"main_category_code": "user_dom_main_category_code", "cnt": "user_dom_main_category_count"})
        entropy = cat_counts.groupby("user_idx")["cnt"].apply(entropy_from_counts).rename("user_category_entropy").reset_index()
        user_stats = user_stats.merge(dom_cat[["user_idx", "user_dom_main_category_code", "user_dom_main_category_count"]], on="user_idx", how="left")
        user_stats = user_stats.merge(entropy, on="user_idx", how="left")
    else:
        user_stats["user_dom_main_category_code"] = -1
        user_stats["user_dom_main_category_count"] = 0
        user_stats["user_category_entropy"] = 0

    store_counts = tmp[tmp["store_code"] >= 0].groupby(["user_idx", "store_code"]).size().reset_index(name="cnt")
    if len(store_counts):
        store_counts = store_counts.sort_values(["user_idx", "cnt"], ascending=[True, False])
        dom_store = store_counts.drop_duplicates("user_idx").rename(columns={"store_code": "user_dom_store_code", "cnt": "user_dom_store_count"})
        user_stats = user_stats.merge(dom_store[["user_idx", "user_dom_store_code", "user_dom_store_count"]], on="user_idx", how="left")
    else:
        user_stats["user_dom_store_code"] = -1
        user_stats["user_dom_store_count"] = 0

    price_stats = tmp.groupby("user_idx").agg(
        user_median_price=("price", "median"),
        user_mean_price=("price", "mean"),
        user_median_rating=("average_rating", "median"),
        user_mean_rating=("average_rating", "mean"),
    ).reset_index()
    user_stats = user_stats.merge(price_stats, on="user_idx", how="left")

    user_stats["user_dom_category_share"] = (
        user_stats["user_dom_main_category_count"].fillna(0) / user_stats["user_history_len"].clip(lower=1)
    ).astype("float32")

    return user_stats

def make_user_histories_sorted(train_df):
    return (
        train_df.sort_values(["user_idx", "timestamp"])
        .groupby("user_idx")["item_idx"]
        .apply(lambda x: x.to_numpy(np.int32))
        .to_dict()
    )

def add_sparse_similarity_features(df, W, train_df, prefix, max_hist=80):
    df = df.reset_index(drop=True)
    W = W.tocsr()
    user_hist = make_user_histories_sorted(train_df)

    sum_arr = np.zeros(len(df), dtype=np.float32)
    mean_arr = np.zeros(len(df), dtype=np.float32)
    max_arr = np.zeros(len(df), dtype=np.float32)
    last_arr = np.zeros(len(df), dtype=np.float32)

    groups = df.groupby("user_idx").indices
    for u, row_pos in tqdm(groups.items(), desc=f"sim-features:{prefix}", leave=False):
        hist = user_hist.get(int(u), np.array([], dtype=np.int32))
        if len(hist) == 0:
            continue
        hist = hist[-max_hist:]
        items = df.loc[row_pos, "item_idx"].to_numpy(np.int32)

        sub = W[hist][:, items]
        if sub.nnz:
            sum_vals = np.asarray(sub.sum(axis=0)).ravel().astype(np.float32)
            try:
                max_vals = np.asarray(sub.max(axis=0).toarray()).ravel().astype(np.float32)
            except Exception:
                max_vals = np.zeros(len(items), dtype=np.float32)
                csc = sub.tocsc()
                for j in range(len(items)):
                    st, en = csc.indptr[j], csc.indptr[j + 1]
                    if en > st:
                        max_vals[j] = csc.data[st:en].max()
            sum_arr[row_pos] = sum_vals
            mean_arr[row_pos] = sum_vals / max(1, len(hist))
            max_arr[row_pos] = max_vals

        last = int(hist[-1])
        last_sim = W[last, items]
        if sparse.issparse(last_sim):
            last_sim = np.asarray(last_sim.toarray()).ravel()
        last_arr[row_pos] = last_sim.astype(np.float32)

    df[f"{prefix}_hist_sim_sum"] = sum_arr
    df[f"{prefix}_hist_sim_mean"] = mean_arr
    df[f"{prefix}_hist_sim_max"] = max_arr
    df[f"{prefix}_last_item_sim"] = last_arr
    return df

def build_candidate_features(candidates, train_context, item_static, sim_feature_mats=None):
    df = candidates.copy().reset_index(drop=True)

    score_cols = [c for c in df.columns if c.endswith("__score")]
    rank_cols = [c for c in df.columns if c.endswith("__rank")]
    for c in score_cols:
        df[c] = df[c].fillna(0).astype("float32")
    for c in rank_cols:
        df[c] = df[c].fillna(1_000_000).astype("float32")
        df[c.replace("__rank", "__rr")] = (1.0 / df[c].clip(lower=1)).astype("float32")

    df["log_n_retrievers"] = np.log1p(df["n_retrievers"].fillna(0)).astype("float32")
    df["log_best_rank"] = np.log1p(df["best_rank"].fillna(1_000_000)).astype("float32")

    item_feat = build_item_context_features(train_context, item_static)
    user_feat = build_user_context_features(train_context, item_static)

    df = df.merge(item_feat, on="item_idx", how="left")
    df = df.merge(user_feat, on="user_idx", how="left")

    df["same_dom_main_category"] = (
        df["main_category_code"].fillna(-999).astype("int32") ==
        df["user_dom_main_category_code"].fillna(-998).astype("int32")
    ).astype("int8")
    df["same_dom_store"] = (
        df["store_code"].fillna(-999).astype("int32") ==
        df["user_dom_store_code"].fillna(-998).astype("int32")
    ).astype("int8")

    df["price_abs_diff_user_median"] = (df["price"] - df["user_median_price"]).abs().astype("float32")
    df["price_abs_diff_user_mean"] = (df["price"] - df["user_mean_price"]).abs().astype("float32")
    df["rating_abs_diff_user_median"] = (df["average_rating"] - df["user_median_rating"]).abs().astype("float32")
    df["rating_abs_diff_user_mean"] = (df["average_rating"] - df["user_mean_rating"]).abs().astype("float32")

    if sim_feature_mats:
        for prefix, W in sim_feature_mats.items():
            df = add_sparse_similarity_features(df, W, train_context, prefix=prefix)

    df["user_id"] = df["user_idx"].map(idx2user).astype("int64")
    df["item_id"] = df["item_idx"].map(idx2item).astype("int64")

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    return df

In [17]:
# -----------------------
# Candidate model fitting and generation
# -----------------------

def fit_candidate_models(train_context, fold_name="full"):
    print(f"\n========== Fitting candidate models for {fold_name} ==========")
    t0 = time.time()
    X = make_interaction_matrix(train_context)

    models = []
    models.extend(build_popularity_models(train_context, item_static, topn=DEFAULT_TOPN_PER_GENERATOR))

    if USE_EASE:
        try:
            models.extend(build_ease_models(X, lambdas=EASE_LAMBDAS, topn=FOLD_MODEL_TOPN))
        except Exception as e:
            print("EASE failed; continuing without it.")
            print(repr(e))

    try:
        models.extend(build_graph_item_models(X))
    except Exception as e:
        print("Graph item models failed; continuing without some/all of them.")
        print(repr(e))

    try:
        models.extend(build_content_models(X, item_static, train_item_indices))
    except Exception as e:
        print("Content KNN failed; continuing without it.")
        print(repr(e))

    try:
        models.extend(build_lightgcn_models(X))
    except Exception as e:
        print("LightGCN failed; continuing without it.")
        print(repr(e))

    try:
        models.extend(build_sasrec_models(train_context))
    except Exception as e:
        print("SASRec failed; continuing without it.")
        print(repr(e))

    print(f"Fitted {len(models)} candidate models for {fold_name} in {(time.time()-t0)/60:.1f} min")
    print([m.name for m in models])
    return models, X

def generate_candidates_for_users(models, user_indices, X_seen, allowed_items, fold_name, chunk_size=256):
    frames = []
    for m in models:
        try:
            f = topk_candidates_from_score_fn(
                m, user_indices, X_seen, allowed_items, chunk_size=chunk_size
            )
            frames.append(f)
            print(f"{fold_name} / {m.name}: {len(f):,} rows")
        except Exception as e:
            print(f"Candidate generation failed for {m.name}; skipping.")
            print(repr(e))

    candidates = combine_candidate_frames(frames)
    print(f"{fold_name}: combined candidates={len(candidates):,}")
    return candidates

def collect_similarity_feature_mats(models, max_mats=4):
    mats = {}
    for m in models:
        if m.use_similarity_features and m.similarity_matrix is not None:
            prefix = m.name.replace("-", "_").replace(".", "_")
            mats[prefix] = m.similarity_matrix
        if len(mats) >= max_mats:
            break
    return mats

In [18]:
# -----------------------
# Fold definitions
# -----------------------

def make_folds():
    folds = []

    if USE_ROLLING_FOLDS:
        rolling_specs = [
            ("val_2020_h1", "2020-01-01", "2020-07-01"),
            ("val_2020_h2", "2020-07-01", "2021-01-01"),
            ("val_2021_q1", "2021-01-01", "2021-04-21"),
        ]
        for name, start, end in rolling_specs:
            start_ts = timestamp_ms(start)
            end_ts = timestamp_ms(end)
            tr = train[train["timestamp"] < start_ts].copy()
            va = train[(train["timestamp"] >= start_ts) & (train["timestamp"] < end_ts)].copy()
            warm = set(tr.user_idx.unique())
            va = va[va.user_idx.isin(warm)].copy()
            if len(tr) and len(va):
                folds.append((name, tr, va))

    folds.append(("provided_test", local_train.copy(), local_valid.copy()))
    return folds

folds = make_folds()
for name, tr, va in folds:
    print(name, "train", tr.shape, "valid", va.shape, "valid users", va.user_idx.nunique())

val_2020_h1 train (110732, 5) valid (9208, 5) valid users 4872
val_2020_h2 train (122770, 5) valid (9881, 5) valid users 5132
val_2021_q1 train (135635, 5) valid (6254, 5) valid users 3664
provided_test train (143313, 5) valid (15158, 5) valid users 7437


In [19]:
# -----------------------
# Build ranker datasets for folds
# -----------------------

def build_fold_reranker_data(fold_name, train_context, valid_context, target_like_only=False, force_rebuild=False):
    cache_path = CACHE_DIR / f"{fold_name}_features.parquet"
    if cache_path.exists() and not force_rebuild:
        print(f"Loading cached fold features: {cache_path}")
        return pd.read_parquet(cache_path)

    valid_users = np.array(sorted(valid_context.user_idx.unique()), dtype=np.int32)
    if target_like_only:
        valid_users = np.array(sorted(set(valid_users) & set(sample_submission.user_idx)), dtype=np.int32)

    if MAX_VALID_USERS_PER_FOLD is not None and len(valid_users) > MAX_VALID_USERS_PER_FOLD:
        rng = np.random.default_rng(SEED)
        valid_users = np.array(sorted(rng.choice(valid_users, size=MAX_VALID_USERS_PER_FOLD, replace=False)), dtype=np.int32)

    valid_context = valid_context[valid_context.user_idx.isin(valid_users)].copy()
    print(f"{fold_name}: users for candidate/ranker data={len(valid_users):,}, positives={len(valid_context):,}")

    models, X_context = fit_candidate_models(train_context, fold_name=fold_name)
    candidates = generate_candidates_for_users(
        models=models,
        user_indices=valid_users,
        X_seen=X_context,
        allowed_items=train_item_indices,
        fold_name=fold_name,
        chunk_size=192,
    )

    positives = set(zip(valid_context.user_idx.astype(int), valid_context.item_idx.astype(int)))
    candidates = prefilter_candidates_for_fold(
        candidates,
        positives=positives,
        max_neg_per_user=MAX_NEGATIVES_PER_USER_FOR_RANKER,
    )

    # Use fewer similarity matrices for features on 64GB; the retriever scores/ranks
    # are still preserved for every generator.
    sim_mats = collect_similarity_feature_mats(models, max_mats=2)
    features = build_candidate_features(candidates, train_context, item_static, sim_feature_mats=sim_mats)

    features["label"] = [
        1 if (int(u), int(i)) in positives else 0
        for u, i in zip(features["user_idx"], features["item_idx"])
    ]
    features["fold"] = fold_name

    cand_pos = features["label"].sum()
    total_pos = len(valid_context.drop_duplicates(["user_idx", "item_idx"]))
    print(f"{fold_name}: candidate positive rows={cand_pos:,}; unique valid positives={total_pos:,}; candidate recall upper bound approx={cand_pos / max(1,total_pos):.4f}")

    saved = save_df(features, cache_path)
    print(f"Saved {fold_name} features to {saved}")

    # Free heavy fitted models and candidate tables before the next fold.
    try:
        del models, X_context, candidates, sim_mats
    except Exception:
        pass
    gc.collect()
    if HAS_TORCH and torch.cuda.is_available():
        torch.cuda.empty_cache()

    return features

# This is intentionally heavy.
# For a quick smoke test, set USE_ROLLING_FOLDS=False and reduce epochs/topn above.
fold_feature_frames = []
for fold_name, tr_ctx, va_ctx in folds:
    feats = build_fold_reranker_data(
        fold_name,
        tr_ctx,
        va_ctx,
        target_like_only=False,
        force_rebuild=False,
    )
    fold_feature_frames.append(feats)

ranker_data = pd.concat(fold_feature_frames, ignore_index=True)
print("Ranker data:", ranker_data.shape)
display(ranker_data[["fold", "user_idx", "item_idx", "label", "n_retrievers", "best_rank"]].head())
save_df(ranker_data, CACHE_DIR / "all_folds_ranker_data.parquet")

val_2020_h1: users for candidate/ranker data=4,872, positives=9,208

========== Fitting candidate models for val_2020_h1 ==========
Fitting ease_lam300: X=(23284, 13441), nnz=110,732, lambda=300.0
Fitting ease_lam1000: X=(23284, 13441), nnz=110,732, lambda=1000.0
Fitting RP3beta alpha=0.7, beta=0.3, topk=300


Fitting RP3beta alpha=1.0, beta=0.5, topk=300


Fitting ItemKNN weighting=cosine, topk=300


Fitting ItemKNN weighting=tfidf, topk=300


Fitting content KNN topk=250


...

provided_test / lgcn_d128_l4_s44: 1,338,660 rows


provided_test / sasrec_d128_l2_s45: 6,660 rows
provided_test: combined candidates=7,949,258
Candidate prefilter for ranker: 7,949,258 -> 1,863,359 rows (kept positives=4,109, max_neg/user=250)


provided_test: candidate positive rows=4,109; unique valid positives=15,158; candidate recall upper bound approx=0.2711
Saved provided_test features to outputs/ensemble_reranker_v2_20260610_220236/cache/provided_test_features.parquet
Ranker data: (5287075, 129)


,fold,user_idx,item_idx,label,n_retrievers,best_rank
0,val_2020_h1,4,5074,1,1,97
1,val_2020_h1,19,4797,1,11,7
2,val_2020_h1,19,12989,1,11,13
3,val_2020_h1,21,10428,1,2,78
4,val_2020_h1,26,10996,1,1,162


PosixPath('outputs/ensemble_reranker_v2_20260610_220236/cache/all_folds_ranker_data.parquet')

In [20]:
# -----------------------
# Train LambdaMART reranker
# -----------------------

assert HAS_LGB, "Install LightGBM with `pip install lightgbm` to train the LambdaMART reranker."

def get_feature_columns(df):
    blocked = {
        "label", "fold",
        "user_id", "item_id", "user_idx", "item_idx",
        "user_first_ts", "user_last_ts", "user_mean_ts",
        "item_first_ts", "item_last_ts", "item_mean_ts",
    }
    feature_cols = []
    for c in df.columns:
        if c in blocked:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feature_cols.append(c)
    return feature_cols

def prepare_lgb_rank_data(df, feature_cols):
    df = df.sort_values(["user_idx", "label"], ascending=[True, False]).reset_index(drop=True)
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(-1).astype(np.float32)
    y = df["label"].astype(np.int32)
    group = df.groupby("user_idx").size().to_numpy()
    return df, X, y, group

if USE_ROLLING_FOLDS and ranker_data["fold"].nunique() > 1:
    train_ranker_df = ranker_data[ranker_data["fold"] != "provided_test"].copy()
    eval_ranker_df = ranker_data[ranker_data["fold"] == "provided_test"].copy()
else:
    users = np.array(sorted(ranker_data.user_idx.unique()))
    rng = np.random.default_rng(SEED)
    eval_users = set(rng.choice(users, size=max(1, int(0.2 * len(users))), replace=False))
    train_ranker_df = ranker_data[~ranker_data.user_idx.isin(eval_users)].copy()
    eval_ranker_df = ranker_data[ranker_data.user_idx.isin(eval_users)].copy()

feature_cols = get_feature_columns(ranker_data)
print("Number of features:", len(feature_cols))
print(feature_cols[:40], "...")

train_sorted, X_tr, y_tr, group_tr = prepare_lgb_rank_data(train_ranker_df, feature_cols)
eval_sorted, X_ev, y_ev, group_ev = prepare_lgb_rank_data(eval_ranker_df, feature_cols)

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=2500,
    learning_rate=0.025,
    num_leaves=127,
    min_child_samples=30,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.2,
    random_state=SEED,
    n_jobs=-1,
    importance_type="gain",
)

ranker.fit(
    X_tr, y_tr,
    group=group_tr,
    eval_set=[(X_ev, y_ev)],
    eval_group=[group_ev],
    eval_at=[10],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150),
        lgb.log_evaluation(period=50),
    ],
)

eval_pred = eval_sorted[["user_idx", "item_idx"]].copy()
eval_pred["score"] = ranker.predict(X_ev, num_iteration=ranker.best_iteration_)

if eval_ranker_df["fold"].nunique() == 1 and eval_ranker_df["fold"].iloc[0] == "provided_test":
    truth_df = local_valid
else:
    truth_pairs = eval_ranker_df[eval_ranker_df.label == 1][["user_idx", "item_idx"]].drop_duplicates()
    truth_df = truth_pairs.assign(timestamp=0, user_id=0, item_id=0)

local_recall = recall_at_k_from_ranked(eval_pred, truth_df, k=10, score_col="score")
print(f"Local held-out reranker Recall@10: {local_recall:.6f}")

if eval_ranker_df["fold"].nunique() == 1 and eval_ranker_df["fold"].iloc[0] == "provided_test":
    target_valid_users = sorted(set(local_valid.user_idx) & set(sample_submission.user_idx))
    target_recall = recall_at_k_from_ranked(eval_pred, local_valid, k=10, users=target_valid_users, score_col="score")
    print(f"Provided-test target-overlap Recall@10: {target_recall:.6f}")

imp = pd.DataFrame({
    "feature": feature_cols,
    "importance_gain": ranker.booster_.feature_importance(importance_type="gain"),
    "importance_split": ranker.booster_.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)
display(imp.head(50))
imp.to_csv(OUT_DIR / "feature_importance.csv", index=False)

ranker.booster_.save_model(str(MODEL_DIR / "lgbm_ranker_validation.txt"))
with open(OUT_DIR / "feature_cols.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

Number of features: 117
['n_retrievers', 'best_rank', 'mean_rank', 'sum_rr', 'mean_rr', 'max_rr', 'content_tfidf_k300__score', 'ease_lam1000__score', 'ease_lam300__score', 'itemknn_cos_k500__score', 'itemknn_tfidf_k500__score', 'lgcn_d128_l3_s42__score', 'lgcn_d128_l4_s44__score', 'lgcn_d256_l3_s43__score', 'pop_category_365d__score', 'pop_decay_180d__score', 'pop_decay_365d__score', 'pop_global__score', 'pop_recent_365d__score', 'pop_recent_730d__score', 'rp3_a07_b03_k500__score', 'rp3_a10_b05_k500__score', 'sasrec_d128_l2_s45__score', 'content_tfidf_k300__rank', 'ease_lam1000__rank', 'ease_lam300__rank', 'itemknn_cos_k500__rank', 'itemknn_tfidf_k500__rank', 'lgcn_d128_l3_s42__rank', 'lgcn_d128_l4_s44__rank', 'lgcn_d256_l3_s43__rank', 'pop_category_365d__rank', 'pop_decay_180d__rank', 'pop_decay_365d__rank', 'pop_global__rank', 'pop_recent_365d__rank', 'pop_recent_730d__rank', 'rp3_a07_b03_k500__rank', 'rp3_a10_b05_k500__rank', 'sasrec_d128_l2_s45__rank'] ...
[LightGBM] [Info] Auto-ch

,feature,importance_gain,importance_split
0,n_retrievers,596728.846433,813
3,sum_rr,178896.518764,1446
74,item_count_last_180d,50622.329890,360
57,log_n_retrievers,35934.368651,109
84,item_days_since_last,26603.236965,990
4,mean_rr,21720.078743,733
1,best_rank,18979.896140,489
92,user_mean_age_days,15474.536665,1027
2,mean_rank,15325.248019,877
46,lgcn_d128_l4_s44__rr,14897.585306,391


In [21]:
# -----------------------
# Train final ranker on all fold data
# -----------------------

final_feature_cols = feature_cols
all_sorted, X_all, y_all, group_all = prepare_lgb_rank_data(ranker_data, final_feature_cols)

best_iter = int(getattr(ranker, "best_iteration_", 1500) or 1500)
final_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=best_iter + 300,
    learning_rate=0.02,
    num_leaves=127,
    min_child_samples=30,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=0.2,
    random_state=SEED + 1,
    n_jobs=-1,
    importance_type="gain",
)
final_ranker.fit(X_all, y_all, group=group_all, eval_at=[10])
final_ranker.booster_.save_model(str(MODEL_DIR / "lgbm_ranker_final.txt"))
print("Final ranker trained on all fold data.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.146205 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 20816
[LightGBM] [Info] Number of data points in the train set: 5287075, number of used features: 117
Final ranker trained on all fold data.


In [22]:
# -----------------------
# Final full-data candidate generation for sample_submission users
# -----------------------

final_models, X_full = fit_candidate_models(train, fold_name="full_train")

for m in final_models:
    m.topn = max(m.topn, FINAL_TOPN_PER_GENERATOR)

target_user_indices = sample_submission["user_idx"].to_numpy(np.int32)

final_candidates = generate_candidates_for_users(
    models=final_models,
    user_indices=target_user_indices,
    X_seen=X_full,
    allowed_items=train_item_indices,
    fold_name="full_train",
    chunk_size=192,
)

final_candidates = prefilter_candidates_for_final(
    final_candidates,
    max_per_user=MAX_FINAL_CANDIDATES_PER_USER,
)

final_sim_mats = collect_similarity_feature_mats(final_models, max_mats=2)
final_features = build_candidate_features(final_candidates, train, item_static, sim_feature_mats=final_sim_mats)
save_df(final_features, CACHE_DIR / "final_candidate_features.parquet")
print("Final feature matrix:", final_features.shape)


========== Fitting candidate models for full_train ==========
Fitting ease_lam300: X=(23284, 13441), nnz=158,471, lambda=300.0
Fitting ease_lam1000: X=(23284, 13441), nnz=158,471, lambda=1000.0
Fitting RP3beta alpha=0.7, beta=0.3, topk=300


Fitting RP3beta alpha=1.0, beta=0.5, topk=300


Fitting ItemKNN weighting=cosine, topk=300


Fitting ItemKNN weighting=tfidf, topk=300


Fitting content KNN topk=250


...

full_train / lgcn_d256_l3_s43: 789,250 rows


full_train / lgcn_d128_l4_s44: 789,250 rows


full_train / sasrec_d128_l2_s45: 4,900 rows


full_train: combined candidates=4,499,871
Final candidate prefilter: 4,499,871 -> 2,706,000 rows (max/user=1200)


Final feature matrix: (2706000, 127)


In [23]:
# -----------------------
# Score final candidates and create submissions
# -----------------------

def align_to_feature_cols(df, feature_cols):
    aligned = df.copy()
    for c in feature_cols:
        if c not in aligned.columns:
            aligned[c] = np.nan
    X = aligned[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(-1).astype(np.float32)
    return X

X_final = align_to_feature_cols(final_features, final_feature_cols)
final_features["rerank_score"] = final_ranker.predict(X_final)

rank_cols = [c for c in final_features.columns if c.endswith("__rank")]
rr_cols = [c.replace("__rank", "__rr") for c in rank_cols if c.replace("__rank", "__rr") in final_features.columns]
if rr_cols:
    final_features["blend_rr_score"] = final_features[rr_cols].sum(axis=1)
else:
    final_features["blend_rr_score"] = 0.0
final_features["blend_rr_score"] += 0.05 * final_features["n_retrievers"].fillna(0)
final_features["blend_rr_score"] -= 1e-6 * final_features["best_rank"].fillna(1_000_000)

fallback_scores = time_weighted_item_scores(train, half_life_days=365)
fallback_order = train_item_indices[np.argsort(-fallback_scores[train_item_indices])]

seen_by_user = {int(u): set(X_full[int(u)].indices.tolist()) for u in sample_submission.user_idx}

def detect_submission_format(sample_sub):
    cols = list(sample_sub.columns)
    if "user_id" not in cols:
        raise ValueError("sample_submission must contain user_id")
    if "item_id" in cols:
        return "single_row_list"
    if "prediction" in cols:
        return "prediction"
    if "ID" in cols:
        return "single_row_list"
    return "single_row_list"

def make_submission_from_scores(scored_df, sample_sub, score_col, out_path, k=10):
    fmt = detect_submission_format(sample_sub)
    scored_df = scored_df.sort_values(["user_idx", score_col, "n_retrievers", "best_rank"],
                                      ascending=[True, False, False, True])
    pred_by_user = (
        scored_df.groupby("user_idx")["item_idx"]
        .apply(lambda x: list(dict.fromkeys([int(v) for v in x.values])))
        .to_dict()
    )

    rows = []
    for _, row in tqdm(sample_sub.iterrows(), total=len(sample_sub), desc=f"submission:{score_col}"):
        uidx = int(row["user_idx"])
        seen = seen_by_user.get(uidx, set())
        preds = []
        for it in pred_by_user.get(uidx, []):
            if it not in seen and it not in preds:
                preds.append(it)
            if len(preds) == k:
                break

        for it in fallback_order:
            it = int(it)
            if it not in seen and it not in preds:
                preds.append(it)
            if len(preds) == k:
                break

        item_ids = [str(int(idx2item[it])) for it in preds[:k]]

        if fmt in ["single_row_list", "prediction"]:
            out_row = {}
            if "ID" in sample_sub.columns:
                out_row["ID"] = row["ID"]
            out_row["user_id"] = int(row["user_id"])
            if "item_id" in sample_sub.columns:
                out_row["item_id"] = ",".join(item_ids)
            elif "prediction" in sample_sub.columns:
                out_row["prediction"] = " ".join(item_ids)
            else:
                out_row["item_id"] = ",".join(item_ids)
            rows.append(out_row)
        else:
            raise ValueError(f"Unsupported submission format: {fmt}")

    sub = pd.DataFrame(rows)
    sub.to_csv(out_path, index=False)
    print(f"Saved {out_path}, shape={sub.shape}")
    return sub

submission_reranked = make_submission_from_scores(
    final_features,
    sample_submission,
    score_col="rerank_score",
    out_path=OUT_DIR / "submission_reranked.csv",
    k=10,
)

submission_blend = make_submission_from_scores(
    final_features,
    sample_submission,
    score_col="blend_rr_score",
    out_path=OUT_DIR / "submission_blend_rr.csv",
    k=10,
)

display(submission_reranked.head())

submission:rerank_score: 100%|██████████| 2255/2255 [10:10<00:00,  3.69it/s]


Saved outputs/ensemble_reranker_v2_20260610_220236/submission_reranked.csv, shape=(2255, 3)


submission:blend_rr_score: 100%|██████████| 2255/2255 [00:00<00:00, 87679.43it/s]

Saved outputs/ensemble_reranker_v2_20260610_220236/submission_blend_rr.csv, shape=(2255, 3)


,ID,user_id,item_id
0,12,12,"796,13149,13033,7048,2990,1683,5964,11495,9436..."
1,14,14,"11590,646,11258,3005,6750,12611,8744,9866,1996..."
2,17,17,"7665,12359,3341,1813,10371,8340,796,36,237,12611"
3,21,21,"796,3197,1533,7157,4053,2875,9190,950,8546,9866"
4,44,44,"796,12394,2169,8703,2990,10694,1180,9306,10340..."


In [24]:
# -----------------------
# Save diagnostics
# -----------------------

diagnostics = {
    "run_id": RUN_ID,
    "n_users": int(n_users),
    "n_items": int(n_items),
    "n_train_rows_dedup": int(len(train)),
    "n_test_rows_dedup": int(len(test)),
    "n_submission_users": int(len(sample_submission)),
    "n_final_candidates": int(len(final_candidates)),
    "n_final_feature_rows": int(len(final_features)),
    "feature_count": int(len(final_feature_cols)),
    "local_validation_recall_at_10": float(local_recall) if "local_recall" in globals() else None,
}
if "target_recall" in globals():
    diagnostics["target_overlap_validation_recall_at_10"] = float(target_recall)

with open(OUT_DIR / "diagnostics.json", "w") as f:
    json.dump(diagnostics, f, indent=2)

print(json.dumps(diagnostics, indent=2))
print("\nKey outputs:")
print(" -", OUT_DIR / "submission_reranked.csv")
print(" -", OUT_DIR / "submission_blend_rr.csv")
print(" -", OUT_DIR / "feature_importance.csv")
print(" -", OUT_DIR / "diagnostics.json")

{
  "run_id": "20260610_220236",
  "n_users": 23284,
  "n_items": 13441,
  "n_train_rows_dedup": 158471,
  "n_test_rows_dedup": 15158,
  "n_submission_users": 2255,
  "n_final_candidates": 2706000,
  "n_final_feature_rows": 2706000,
  "feature_count": 117,
  "local_validation_recall_at_10": 0.15147215445722909,
  "target_overlap_validation_recall_at_10": 0.14436816750046322
}

Key outputs:
 - outputs/ensemble_reranker_v2_20260610_220236/submission_reranked.csv
 - outputs/ensemble_reranker_v2_20260610_220236/submission_blend_rr.csv
 - outputs/ensemble_reranker_v2_20260610_220236/feature_importance.csv
 - outputs/ensemble_reranker_v2_20260610_220236/diagnostics.json


## Practical tuning checklist

After the first full run:

1. Submit both `submission_reranked.csv` and `submission_blend_rr.csv`.
2. Check `feature_importance.csv`. If model scores dominate too much, add more candidate diversity.
3. Tune in this order:
   - EASE λ values
   - RP3β α/β/topK
   - LightGCN seed/layers/dimensions
   - Candidate `topn`
   - LightGBM `num_leaves`, `min_child_samples`, and `learning_rate`
4. Use the local `provided_test` target-overlap Recall@10 as your main sanity check.
5. For leaderboard pushes, submit variants:
   - final ranker trained on all folds
   - ranker excluding oldest fold
   - reciprocal-rank blend
   - reranker plus a small recent-popularity boost